# Projet fil rouge, partie 2 : quantification

Ce notebook est une **amorce** : il fonctionne tel quel sur un réseau de démonstration. Votre
travail : l'appliquer à **votre** réseau du fil rouge, puis explorer. La consigne et les pistes :
<https://bonben.github.io/ensta-tp-deep-learning/05c_quantification/>.

Sur Colab : *Exécution > Modifier le type d'exécution > GPU T4*.

## Les données

In [ ]:
import os
import tarfile
import urllib.request

CIFAR_URL = "https://github.com/bonben/ensta-tp-deep-learning/releases/download/cifar10/cifar-10-python.tar.gz"

if not os.path.exists("data/cifar-10-batches-py"):
    urllib.request.urlretrieve(CIFAR_URL, "cifar.tar.gz")
    with tarfile.open("cifar.tar.gz") as tar:
        tar.extractall("data", filter="data")
    os.remove("cifar.tar.gz")
print("CIFAR-10 est prêt")

## Les outils

Ceux de la boîte à outils du fil rouge, plus `count_nonzero(model)`. `fit` accepte maintenant
une perte à vous (`loss_fn`), dont se sert la distillation.

In [ ]:
import copy
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.transforms import v2
from torch.utils.flop_counter import FlopCounterMode
import matplotlib.pyplot as plt

torch.multiprocessing.set_start_method("fork", force=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Calcul sur :", device)

MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
NORMALIZE = [v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)]

transform_train = v2.Compose([v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip()] + NORMALIZE)
transform_val = v2.Compose(NORMALIZE)
train_set = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=transform_train)
val_set = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=transform_val)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=256, shuffle=False, num_workers=2)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def count_nonzero(model):
    """Paramètres non nuls (après pruning, les poids mis à zéro ne comptent plus)."""
    total = 0
    for m in model.modules():
        for name, p in m.named_parameters(recurse=False):
            if name.endswith("_orig"):  # poids pruné : on compte le poids masqué
                p = getattr(m, name[:-len("_orig")])
            total += int((p != 0).sum())
    return total


def count_macs(model):
    model.eval()
    with FlopCounterMode(display=False) as counter:
        model(torch.zeros(1, 3, 32, 32, device=next(model.parameters()).device))
    return counter.get_total_flops() // 2


def evaluate(model, loader=val_loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            correct += (model(inputs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total


def fit(model, epochs, lr=0.1, weight_decay=5e-4, batch_size=128, seed=0, name="essai", loss_fn=None):
    """Entraîne avec SGD (momentum 0.9) et un scheduler cosinus. Renvoie l'historique.

    loss_fn(outputs, labels, inputs) remplace la cross-entropy si on la donne (distillation).
    """
    torch.manual_seed(seed)
    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    model = model.to(device)
    if loss_fn is None:
        loss_fn = lambda outputs, labels, inputs: F.cross_entropy(outputs, labels)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {"train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        correct, total = 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, labels, inputs)
            loss.backward()
            optimizer.step()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        scheduler.step()
        history["train_acc"].append(100 * correct / total)
        history["val_acc"].append(evaluate(model))
        print(f"époque {epoch + 1}/{epochs} | train {history['train_acc'][-1]:.1f} % | "
              f"val {history['val_acc'][-1]:.1f} % | {time.time() - t0:.0f} s")
    print(f"\nRÉSUMÉ : {name} | {count_params(model)} paramètres ({count_nonzero(model)} non nuls) | {epochs} époques | "
          f"val au dernier epoch {history['val_acc'][-1]:.2f} %")
    return history


def plot(*histories, labels=None):
    """Superpose les courbes de validation de plusieurs essais."""
    for i, h in enumerate(histories):
        plt.plot(h["val_acc"], label=labels[i] if labels else f"essai {i + 1}")
    plt.xlabel("époque"); plt.ylabel("précision de validation (%)")
    plt.grid(alpha=0.3); plt.legend(); plt.show()

## Le réseau

Le ResNet du fil rouge et sa version factorisée (cours 3). **Remplacez ou complétez cette cellule
par la classe de votre réseau.**

In [ ]:
class BasicBlock(nn.Module):
    """Deux convolutions 3×3 et un raccourci (la connexion résiduelle)."""

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)


class SeparableConv2d(nn.Module):
    """Convolution séparable : depthwise 3×3 puis pointwise 1×1 (cours 3)."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch, bias=False)
        self.pointwise = nn.Conv2d(in_ch, out_ch, 1, bias=False)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class SepBasicBlock(BasicBlock):
    def __init__(self, in_planes, planes, stride=1):
        super().__init__(in_planes, planes, stride)
        self.conv1 = SeparableConv2d(in_planes, planes, stride)
        self.conv2 = SeparableConv2d(planes, planes, 1)


class ResNet(nn.Module):
    """4 étages ; le nombre de FM vaut width, 2·width, 4·width, 8·width."""

    def __init__(self, block, num_blocks, num_classes=10, width=64):
        super().__init__()
        self.in_planes = width
        self.conv1 = nn.Conv2d(3, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(width)
        self.layer1 = self._make_layer(block, width, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 2 * width, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 4 * width, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 8 * width, num_blocks[3], stride=2)
        self.linear = nn.Linear(8 * width, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        layers = []
        for s in [stride] + [1] * (num_blocks - 1):
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.linear(out)

## Votre réseau de départ

Par défaut, le réseau de démonstration : le ResNet18 factorisé, `width=32` (371 658 paramètres,
environ 89 % en 15 époques). Mettez le vôtre à la place, et rechargez ses poids si vous les avez
sauvegardés : pas besoin de l'entraîner.

In [ ]:
model = ResNet(SepBasicBlock, [2, 2, 2, 2], width=32).to(device)   # <- votre réseau
print(f"{count_params(model):,} paramètres, {count_macs(model):,} MAC")

# Vos poids sauvegardés dans Drive (décommentez) :
# from google.colab import drive
# drive.mount("/content/drive")
# model.load_state_dict(torch.load("/content/drive/MyDrive/mon_reseau.pt", map_location=device))
# print(f"précision : {evaluate(model):.2f} %")

In [ ]:
# Sinon, entraînez-le ici (une fois), puis sauvegardez-le.
h_ref = fit(model, epochs=15, name="réseau de départ")
torch.save(model.state_dict(), "reseau_depart.pt")

## La quantification

`quantize(w, bits)` arrondit chaque poids sur une grille de $2^{b} - 1$ valeurs, symétrique
autour de 0, avec une échelle par filtre. `quantize_model(model, bits)` l'applique à toutes les
convolutions et à la couche linéaire d'une **copie** du réseau. Les calculs restent en flottant
(on simule la quantification), mais les poids ne prennent plus que $2^{b} - 1$ valeurs.

`model_size_kb(model, bits)` donne la taille mémoire du réseau : poids sur `bits` bits,
BatchNorm et biais en 32 bits.

In [ ]:
import torch.nn.utils.parametrize as parametrize


def quantize(w, bits):
    """Quantification symétrique, une échelle par filtre : 2^(bits-1) - 1 niveaux de chaque côté de 0."""
    qmax = 2 ** (bits - 1) - 1
    dims = tuple(range(1, w.dim()))
    scale = w.detach().abs().amax(dim=dims, keepdim=True).clamp(min=1e-8) / qmax
    return torch.round(w / scale).clamp(-qmax, qmax) * scale


class FakeQuant(nn.Module):
    """Remplace le poids par sa version quantifiée, sans bloquer le gradient (straight-through)."""

    def __init__(self, bits):
        super().__init__()
        self.bits = bits

    def forward(self, w):
        return w + (quantize(w, self.bits) - w).detach()


def quantize_model(model, bits):
    """Quantifie les poids de toutes les convolutions et de la couche linéaire (sur une copie)."""
    model = copy.deepcopy(model)
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            parametrize.register_parametrization(m, "weight", FakeQuant(bits))
    return model


def model_size_kb(model, bits):
    """Taille mémoire : les poids quantifiés sur `bits` bits, le reste (BatchNorm, biais) en 32 bits."""
    total = 0
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            total += m.weight.numel() * bits + (m.bias.numel() * 32 if m.bias is not None else 0)
        elif isinstance(m, nn.BatchNorm2d):
            total += 2 * m.num_features * 32
    return total / 8 / 1024

## Quantifier après l'entraînement

Prédisez d'abord : à partir de combien de bits la précision va-t-elle s'effondrer ?

In [ ]:
print(f"32 bits : {model_size_kb(model, 32):.0f} Ko, {evaluate(model):.2f} %")
for bits in [8, 6, 4, 3, 2]:
    q = quantize_model(model, bits)
    print(f"{bits} bits : {model_size_kb(q, bits):.0f} Ko, {evaluate(q):.2f} %")

## Fine-tuner en quantifiant

Le réseau quantifié se fine-tune quelques époques. Dans le calcul, les poids sont quantifiés ;
le gradient, lui, traverse l'arrondi comme s'il n'existait pas (*straight-through estimator*) et
met à jour des poids en flottant.

In [ ]:
q = quantize_model(model, 3)
print(f"3 bits, sans fine-tuning : {evaluate(q):.2f} %")
h_qat = fit(q, epochs=5, lr=0.01, name="3 bits + 5 époques")
print(f"{model_size_kb(q, 3):.0f} Ko")

## À vous

- jusqu'à combien de bits descendez-vous avec un fine-tuning ? À 2 bits, les poids sont
  ternaires (-1, 0 ou +1, fois l'échelle du filtre) ;
- quelles couches sont les plus sensibles ? Gardez par exemple la **première et la dernière**
  couche sur 8 bits (précision mixte) ;
- une échelle **par couche** au lieu d'une par filtre : qu'est-ce que ça change ?
- les **activations** restent en flottant : comment les quantifier, et qu'y gagnerait-on ?

In [ ]:
# q = quantize_model(model, ...)
# ...